# Day 09. Exercise 03
# Ensembles

## 0. Imports

In [1]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, BaggingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score

## 1. Preprocessing

1. Create the same dataframe as in the previous exercise.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test` and then get `X_train`, `y_train`, `X_valid`, `y_valid` from the previous `X_train`, `y_train`. Use the additional parameter `stratify`.

1. Создайте тот же фрейм данных, что и в предыдущем упражнении.
2. Используя `train_test_split` с параметрами `test_size=0.2`, `random_state=21`, получаем `X_train`, `y_train`, `X_test`, `y_test", а затем получаем `X_train`, `y_train`, `X_valid`, `y_valid` из предыдущего `X_train`, `y_train`. Используйте дополнительный параметр `стратифицировать`.

In [2]:
df_features = pd.read_csv('../data/day-of-week-not-scaled.csv')
df_target = pd.read_csv('../data/dayofweek.csv')

In [3]:
df = df_features.copy()
df['dayofweek'] = df_target['dayofweek']

X = df.drop('dayofweek', axis=1)
y = df['dayofweek']

In [4]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=21, stratify=y
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=21, stratify=y_train_full
)

print(f"Train shape: {X_train.shape}")
print(f"Valid shape: {X_valid.shape}")
print(f"Test shape: {X_test.shape}")

Train shape: (1078, 43)
Valid shape: (270, 43)
Test shape: (338, 43)


## 2. Individual classifiers

1. Train SVM, decision tree and random forest again with the best parameters that you got from the 01 exercise with `random_state=21` for all of them.
2. Evaluate `accuracy`, `precision`, and `recall` for them on the validation set.
3. The result of each cell of the section should look like this:

```
accuracy is 0.87778
precision is 0.88162
recall is 0.87778
```

1. Снова обучите SVM, дерево решений и случайный лес наилучшим параметрам, которые вы получили в упражнении 01, указав для них "random_state=21".
2. Оцените "точность", "прецизионность" и "отзыв" для них в наборе проверки.
3. Результат работы каждой ячейки раздела должен выглядеть следующим образом:
```
accuracy is 0.87778
precision is 0.88162
recall is 0.87778
```

In [5]:
def print_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted')
    rec = recall_score(y_true, y_pred, average='weighted')
    print(f"accuracy is {acc:.5f}")
    print(f"precision is {prec:.5f}")
    print(f"recall is {rec:.5f}")
    return acc, prec

In [6]:
svc = SVC(C=10, kernel='rbf', gamma='scale', class_weight='balanced', probability=True, random_state=21)
svc.fit(X_train, y_train)
print("--- SVM (Validation) ---")
print_metrics(y_valid, svc.predict(X_valid))

--- SVM (Validation) ---
accuracy is 0.39259
precision is 0.48363
recall is 0.39259


(0.3925925925925926, 0.4836278520040513)

In [7]:
dt = DecisionTreeClassifier(max_depth=25, criterion='entropy', class_weight='balanced', random_state=21)
dt.fit(X_train, y_train)
print("--- Decision Tree (Validation) ---")
print_metrics(y_valid, dt.predict(X_valid))

--- Decision Tree (Validation) ---
accuracy is 0.85185
precision is 0.85481
recall is 0.85185


(0.8518518518518519, 0.8548103632478633)

In [8]:
rf = RandomForestClassifier(n_estimators=100, max_depth=25, criterion='entropy', random_state=21)
rf.fit(X_train, y_train)
print("--- Random Forest (Validation) ---")
print_metrics(y_valid, rf.predict(X_valid))

--- Random Forest (Validation) ---
accuracy is 0.90370
precision is 0.90328
recall is 0.90370


(0.9037037037037037, 0.903284088974337)

## 3. Voting classifiers

1. Using `VotingClassifier` and the three models that you have just trained, calculate the `accuracy`, `precision`, and `recall` on the validation set.
2. Play with the other parameteres.
3. Calculate the `accuracy`, `precision` and `recall` on the test set for the model with the best weights in terms of accuracy (if there are several of them with equal values, choose the one with the higher precision).

1. Используя "VotingClassifier" и три модели, которые вы только что обучили, рассчитайте "точность", "прецизионность" и "отзыв" в наборе проверки.
2. Поиграйте с другими параметрами.
3. Рассчитайте "точность", "прецизионность" и "отзыв" в тестовом наборе для модели с наилучшими весами с точки зрения точности (если есть несколько весов с одинаковыми значениями, выберите тот, который имеет более высокую точность).

In [9]:
estimators = [('svc', svc), ('dt', dt), ('rf', rf)]

voting_hard = VotingClassifier(estimators=estimators, voting='hard')
voting_hard.fit(X_train, y_train)
print("--- Voting Hard (Validation) ---")
print_metrics(y_valid, voting_hard.predict(X_valid))

--- Voting Hard (Validation) ---
accuracy is 0.87037
precision is 0.87196
recall is 0.87037


(0.8703703703703703, 0.8719639842411735)

In [10]:
weight_combinations = [
    [1, 1, 1],
    [2, 1, 1],
    [1, 2, 1],
    [1, 1, 2],
    [2, 2, 1],
    [1, 2, 2]
]

best_voting_acc = -1
best_voting_prec = -1
best_weights = None
best_voting_model = None

print("\n--- Tuning Voting Weights ---")
for w in weight_combinations:
    v_clf = VotingClassifier(estimators=estimators, voting='soft', weights=w)
    v_clf.fit(X_train, y_train)
    y_val_pred = v_clf.predict(X_valid)
    
    acc = accuracy_score(y_valid, y_val_pred)
    prec = precision_score(y_valid, y_val_pred, average='weighted')
    
    if acc > best_voting_acc:
        best_voting_acc = acc
        best_voting_prec = prec
        best_weights = w
        best_voting_model = v_clf
    elif acc == best_voting_acc:
        if prec > best_voting_prec:
            best_voting_prec = prec
            best_weights = w
            best_voting_model = v_clf

print(f"Best Weights: {best_weights} with Acc: {best_voting_acc:.5f}")


--- Tuning Voting Weights ---
Best Weights: [1, 1, 2] with Acc: 0.86667


In [11]:
print("\n--- Best Voting Classifier (Test Set) ---")
y_test_pred_voting = best_voting_model.predict(X_test)
print_metrics(y_test, y_test_pred_voting)


--- Best Voting Classifier (Test Set) ---
accuracy is 0.89645
precision is 0.89678
recall is 0.89645


(0.8964497041420119, 0.8967839833256478)

## 4. Bagging classifiers

1. Using `BaggingClassifier` and `SVM` with the best parameters create an ensemble, try different values of the `n_estimators`, use `random_state=21`.
2. Play with the other parameters.
3. Calculate the `accuracy`, `precision`, and `recall` for the model with the best parameters (in terms of accuracy) on the test set (if there are several of them with equal values, choose the one with the higher precision)

1. Используя "BaggingClassifier" и "SVM" с наилучшими параметрами, создайте ансамбль, попробуйте разные значения "n_estimators", используйте "random_state=21`.
2. Поиграйте с другими параметрами.
3. Рассчитайте "точность", "прецизионность" и "отзыв" для модели с наилучшими параметрами (с точки зрения точности) в тестовом наборе (если их несколько с одинаковыми значениями, выберите модель с более высокой точностью).

In [12]:
n_estimators_list = [10, 20, 50]
best_bag_acc = -1
best_bag_prec = -1
best_bag_est = None
best_bag_model = None

print("--- Tuning Bagging n_estimators ---")
for n in n_estimators_list:
    bag = BaggingClassifier(estimator=svc, n_estimators=n, random_state=21, n_jobs=-1)
    bag.fit(X_train, y_train)
    
    y_val_pred = bag.predict(X_valid)
    acc = accuracy_score(y_valid, y_val_pred)
    prec = precision_score(y_valid, y_val_pred, average='weighted')
    
    print(f"n_estimators={n} | Acc: {acc:.5f}")
    
    if acc > best_bag_acc:
        best_bag_acc = acc
        best_bag_prec = prec
        best_bag_est = n
        best_bag_model = bag
    elif acc == best_bag_acc:
        if prec > best_bag_prec:
            best_bag_prec = prec
            best_bag_est = n
            best_bag_model = bag

print(f"Best Bagging n_estimators: {best_bag_est}")

--- Tuning Bagging n_estimators ---


/Users/dinarvaliev/micromamba/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


n_estimators=10 | Acc: 0.44074


/Users/dinarvaliev/micromamba/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


n_estimators=20 | Acc: 0.46296
n_estimators=50 | Acc: 0.43704
Best Bagging n_estimators: 20


/Users/dinarvaliev/micromamba/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [13]:
print("\n--- Best Bagging Classifier (Test Set) ---")
y_test_pred_bag = best_bag_model.predict(X_test)
print_metrics(y_test, y_test_pred_bag)


--- Best Bagging Classifier (Test Set) ---
accuracy is 0.48817
precision is 0.43236
recall is 0.48817


/Users/dinarvaliev/micromamba/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


(0.4881656804733728, 0.43235793626077024)

## 5. Stacking classifiers

1. To achieve reproducibility in this case you will have to create an object of cross-validation generator: `StratifiedKFold(n_splits=n, shuffle=True, random_state=21)`, where `n` you will try to optimize (the details are below).
2. Using `StackingClassifier` and the three models that you have recently trained, calculate the `accuracy`, `precision` and `recall` on the validation set, try different values of `n_splits` `[2, 3, 4, 5, 6, 7]` in the cross-validation generator and parameter `passthrough` in the classifier itself,
3. Calculate the `accuracy`, `precision`, and `recall` for the model with the best parameters (in terms of accuracy) on the test set (if there are several of them with equal values, choose the one with the higher precision). Use `final_estimator=LogisticRegression(solver='liblinear')`.

1. Для достижения воспроизводимости в этом случае вам нужно будет создать объект генератора перекрестной проверки: `StratifiedKFold(n_splits=n, shuffle=True, random_state=21)`, где `n` вы попытаетесь оптимизировать (подробности приведены ниже).
2. Используя "StackingClassifier" и три модели, которые вы недавно обучили, рассчитайте "точность", "precision" и "recall" для набора проверки, попробуйте разные значения "n_splits"` `[2, 3, 4, 5, 6, 7]` в перекрестномгенератор проверки и параметр `passthrough` в самом классификаторе,
3. Рассчитайте `точность", "прецизионность" и "отзыв" для модели с наилучшими параметрами (с точки зрения точности) в тестовом наборе (если их несколько с одинаковыми значениями, выберите модель с более высокой точностью). Используйте `final_estimator=LogisticRegression(solver='liblinear')`.

In [14]:
n_splits_list = [2, 3, 4, 5, 6, 7]
passthrough_list = [False, True]

best_stack_acc = -1
best_stack_prec = -1
best_stack_params = {}
best_stack_model = None

final_est = LogisticRegression(solver='liblinear')

print("--- Tuning Stacking ---")
for n in n_splits_list:
    cv = StratifiedKFold(n_splits=n, shuffle=True, random_state=21)
    
    for pt in passthrough_list:
        stack = StackingClassifier(
            estimators=estimators,
            final_estimator=final_est,
            cv=cv,
            passthrough=pt,
            n_jobs=-1
        )
        stack.fit(X_train, y_train)
        
        y_val_pred = stack.predict(X_valid)
        acc = accuracy_score(y_valid, y_val_pred)
        prec = precision_score(y_valid, y_val_pred, average='weighted')
        
        print(f"splits={n}, passthrough={pt} | Acc: {acc:.5f}")
        
        if acc > best_stack_acc:
            best_stack_acc = acc
            best_stack_prec = prec
            best_stack_params = {'n_splits': n, 'passthrough': pt}
            best_stack_model = stack
        elif acc == best_stack_acc:
            if prec > best_stack_prec:
                best_stack_prec = prec
                best_stack_params = {'n_splits': n, 'passthrough': pt}
                best_stack_model = stack

print(f"Best Stacking Params: {best_stack_params}")

--- Tuning Stacking ---
splits=2, passthrough=False | Acc: 0.89259
splits=2, passthrough=True | Acc: 0.89259
splits=3, passthrough=False | Acc: 0.89259
splits=3, passthrough=True | Acc: 0.90000
splits=4, passthrough=False | Acc: 0.89259
splits=4, passthrough=True | Acc: 0.88889
splits=5, passthrough=False | Acc: 0.89630
splits=5, passthrough=True | Acc: 0.88889
splits=6, passthrough=False | Acc: 0.89259
splits=6, passthrough=True | Acc: 0.89630
splits=7, passthrough=False | Acc: 0.88519
splits=7, passthrough=True | Acc: 0.90000
Best Stacking Params: {'n_splits': 3, 'passthrough': True}


In [15]:
print("\n--- Best Stacking Classifier (Test Set) ---")
y_test_pred_stack = best_stack_model.predict(X_test)
print_metrics(y_test, y_test_pred_stack)


--- Best Stacking Classifier (Test Set) ---
accuracy is 0.90533
precision is 0.90537
recall is 0.90533


(0.9053254437869822, 0.9053665536873795)

## 6. Predictions

1. Choose the best model in terms of accuracy (if there are several of them with equal values, choose the one with the higher precision).
2. Analyze: for which weekday your model makes the most errors (in % of the total number of samples of that class in your full dataset), for which labname and for which users.
3. Save the model.

1. Выберите наилучшую модель с точки зрения точности (если есть несколько моделей с одинаковыми значениями, выберите ту, которая имеет более высокую точность).
2. Проанализируйте: для какого дня недели ваша модель допускает наибольшее количество ошибок (в % от общего количества выборок этого класса в вашем полном наборе данных), для какой лаборатории и для каких пользователей.
3. Сохраните модель.

In [16]:
acc_voting = accuracy_score(y_test, y_test_pred_voting)
prec_voting = precision_score(y_test, y_test_pred_voting, average='weighted')

acc_bagging = accuracy_score(y_test, y_test_pred_bag)
prec_bagging = precision_score(y_test, y_test_pred_bag, average='weighted')

acc_stacking = accuracy_score(y_test, y_test_pred_stack)
prec_stacking = precision_score(y_test, y_test_pred_stack, average='weighted')

print(f"\nFinal Comparison (Test Set):")
print(f"Voting:   Acc={acc_voting:.5f}, Prec={prec_voting:.5f}")
print(f"Bagging:  Acc={acc_bagging:.5f}, Prec={prec_bagging:.5f}")
print(f"Stacking: Acc={acc_stacking:.5f}, Prec={prec_stacking:.5f}")

models = [
    (best_voting_model, acc_voting, prec_voting, "Voting"),
    (best_bag_model, acc_bagging, prec_bagging, "Bagging"),
    (best_stack_model, acc_stacking, prec_stacking, "Stacking")
]

models.sort(key=lambda x: (x[1], x[2]), reverse=True)
champion_model = models[0][0]
print(f"\nChampion Model: {models[0][3]}")


Final Comparison (Test Set):
Voting:   Acc=0.89645, Prec=0.89678
Bagging:  Acc=0.48817, Prec=0.43236
Stacking: Acc=0.90533, Prec=0.90537

Champion Model: Stacking


/Users/dinarvaliev/micromamba/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [17]:
y_pred_final = champion_model.predict(X_test)
errors_mask = y_test != y_pred_final
error_classes = y_test[errors_mask]

full_dataset_counts = df['dayofweek'].value_counts()
error_counts = error_classes.value_counts()

weekday_error_rates = (error_counts / full_dataset_counts * 100).fillna(0).sort_values(ascending=False)

print("\nError analysis (Error count / Total count in full dataset %):")
print(weekday_error_rates)
print(f"Weekday with most errors: {weekday_error_rates.idxmax()}")

X_test_errors = X_test.loc[errors_mask.index]

lab_cols = [c for c in X.columns if c.startswith('labname_')]
error_labs = X_test_errors[lab_cols].idxmax(axis=1)
print(f"Labname with most errors: {error_labs.mode()[0]}")

user_cols = [c for c in X.columns if c.startswith('uid_user_')]
error_users = X_test_errors[user_cols].idxmax(axis=1)
print(f"User with most errors: {error_users.mode()[0]}")


Error analysis (Error count / Total count in full dataset %):
dayofweek
0    6.617647
4    2.884615
5    2.214022
1    1.459854
6    1.404494
2    1.342282
3    0.757576
Name: count, dtype: float64
Weekday with most errors: 0
Labname with most errors: labname_project1
User with most errors: uid_user_14


In [18]:
filename = 'best_model.joblib'
joblib.dump(champion_model, filename)
print(f"Model saved to {filename}")

Model saved to best_model.joblib
